# Chapter 3 — Gender and Market Segmentation

Gender-specific catalogue and accord-combination analysis across Men, Women and Unisex classifications. Median segmentation is calculated independently within each classification and analytical level.

Votes indicate platform activity, not sales, market share or unique consumers. Ratings are community evaluations. Catalogue classifications are not evidence of innate preferences.

## 3.1 Configuration and source validation

Raw Fragrantica data supplies ratings, vote counts, classification and ordered accords. Notebook 13's prepared table supplies separately cleaned fragrance and brand names. The join uses URL-derived fragrance_id.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from pathlib import Path

SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
REFERENCE_TABLE = "main.fragrance_analytics.fragrantica_prepared"
OUTPUT_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/chapter_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_ORDER = ["Men", "Women", "Unisex"]
SEGMENT_ORDER = ["Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"]
MIN_DOMINANT_FRAGRANCES = 50
MIN_DYAD_FRAGRANCES = 50
MIN_TRIAD_FRAGRANCES = 50

df_source = spark.table(SOURCE_TABLE)
df_reference = spark.table(REFERENCE_TABLE)
print("Source rows:", f"{df_source.count():,}")
print("Prepared reference rows:", f"{df_reference.count():,}")
df_source.printSchema()
df_reference.printSchema()

source_id_stats = df_source.agg(F.count("*").alias("rows"), F.countDistinct("id").alias("unique_ids"))
source_id_check = source_id_stats.first()
assert source_id_check["rows"] == source_id_check["unique_ids"], "Raw source must contain one row per id before deduplication."

df_source_keyed = df_source.withColumn(
    "fragrance_id",
    F.regexp_extract("url", r"-([0-9]+)\.html(?:\?.*)?$", 1)
)
assert df_reference.groupBy("fragrance_id").count().filter(F.col("count") > 1).limit(1).count() == 0, "Duplicate reference fragrance_id values."

df_names = df_reference.select(
    "fragrance_id",
    F.col("fragrance_name_clean").alias("fragrance_name"),
    F.col("brand_clean").alias("brand")
)
df_joined = df_source_keyed.join(df_names, "fragrance_id", "left")
source_rows = df_source_keyed.count()
joined_rows = df_joined.count()
assert joined_rows == source_rows, f"Reference join changed row count: {source_rows} -> {joined_rows}"
print("Join rows preserved:", f"{joined_rows:,}")
print("Missing cleaned names:", df_joined.filter(F.col("fragrance_name").isNull() | (F.trim("fragrance_name") == "")).count())
print("Missing cleaned brands:", df_joined.filter(F.col("brand").isNull() | (F.trim("brand") == "")).count())

df_accords = (
    df_joined
    .withColumn("accord_array", F.from_json(
        F.regexp_replace(F.regexp_replace(F.col("main_accords"), "'", '"'), r"\bNone\b", "null"),
        "array<string>"
    ))
    .withColumn("gender_clean", F.initcap(F.lower(F.trim("gender"))))
    .filter(F.col("gender_clean").isin(GENDER_ORDER))
    .withColumn("accord_array", F.expr("filter(transform(accord_array, x -> lower(trim(x))), x -> x is not null and x <> '')"))
    .filter(F.col("id").isNotNull() & (F.size("accord_array") > 0))
    .dropDuplicates(["id"])
)
assert df_accords.groupBy("id").count().filter(F.col("count") > 1).limit(1).count() == 0
print("Classified fragrances with valid accords:", f"{df_accords.count():,}")
display(df_accords.groupBy("gender_clean").agg(F.countDistinct("id").alias("fragrance_count")).orderBy("gender_clean"))

## 3.2 Catalogue distribution by classification

Accord prevalence counts unique fragrance–accord associations across all recorded positions. Fragrances may contribute to multiple accords, so percentages are not additive. No fragrance examples are selected in this section.

In [0]:
gender_distribution_ch3 = (
    df_accords.groupBy("gender_clean")
    .agg(F.countDistinct("id").alias("fragrance_count"))
)
total_gendered_ch3 = gender_distribution_ch3.agg(F.sum("fragrance_count")).first()[0]
gender_distribution_ch3 = (
    gender_distribution_ch3
    .withColumn("catalogue_share_pct", F.round(F.col("fragrance_count") / F.lit(total_gendered_ch3) * 100, 2))
    .orderBy("gender_clean")
)
display(gender_distribution_ch3)

df_all_accords_ch3 = (
    df_accords.select("id", "gender_clean", F.explode("accord_array").alias("accord"))
    .dropDuplicates(["id", "gender_clean", "accord"])
)
gender_denominators_ch3 = df_accords.groupBy("gender_clean").agg(F.countDistinct("id").alias("classification_fragrances"))
accord_prevalence_ch3 = (
    df_all_accords_ch3.groupBy("gender_clean", "accord")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .join(gender_denominators_ch3, "gender_clean")
    .withColumn("prevalence_pct", F.round(F.col("fragrance_count") / F.col("classification_fragrances") * 100, 2))
    .orderBy("gender_clean", F.desc("prevalence_pct"), "accord")
)
display(accord_prevalence_ch3)

top_accords_ch3 = [
    r["accord"] for r in accord_prevalence_ch3.groupBy("accord")
    .agg(F.sum("fragrance_count").alias("n"))
    .orderBy(F.desc("n"), "accord").limit(30).collect()
]
display(
    accord_prevalence_ch3.filter(F.col("accord").isin(top_accords_ch3))
    .groupBy("accord").pivot("gender_clean", GENDER_ORDER)
    .agg(F.first("prevalence_pct")).orderBy("accord")
)

In [0]:
from pyspark.sql import functions as F

display(
    df_reference
    .filter(
        F.lower(F.trim(F.col("brand_clean"))).isin(
            "hermes", "lancome", "lancôme"
        )
    )
    .select(
        "fragrance_id",
        "fragrance_name_clean",
        "brand_clean"
    )
    .orderBy("brand_clean", "fragrance_name_clean")
)

In [0]:
from pyspark.sql import functions as F

# Full-dataset validation of prepared fragrance names and brands

name_quality_ch3 = (
    df_accords
    .withColumn(
        "brand_in_name",
        F.when(
            F.col("brand").isNotNull()
            & F.col("fragrance_name").isNotNull(),
            F.instr(
                F.lower(F.col("fragrance_name")),
                F.lower(F.trim(F.col("brand")))
            ) > 0
        ).otherwise(False)
    )
)

quality_summary_ch3 = name_quality_ch3.agg(
    F.count("*").alias("total_fragrances"),
    F.sum(
        F.when(F.col("fragrance_name").isNull(), 1).otherwise(0)
    ).alias("missing_names"),
    F.sum(
        F.when(F.col("brand").isNull(), 1).otherwise(0)
    ).alias("missing_brands"),
    F.sum(
        F.when(F.col("brand_in_name"), 1).otherwise(0)
    ).alias("brand_embedded_in_name"),
    F.countDistinct("id").alias("unique_fragrance_ids")
)

display(quality_summary_ch3)

# Inspect all cases where the brand occurs inside the fragrance name
display(
    name_quality_ch3
    .filter(F.col("brand_in_name"))
    .select("id", "fragrance_name", "brand")
    .orderBy("brand", "fragrance_name")
)

display(
    name_quality_ch3
    .filter(
        F.lower(F.col("brand")).isin("lancome", "lancôme", "hermes", "hermès")
    )
    .select("id", "fragrance_name", "brand")
    .orderBy("brand", "fragrance_name")
)

## 3.3 Shared gender-specific segmentation function

Attention is mean vote count per fragrance; evaluation is mean fragrance rating. Medians are calculated independently within each classification and analytical level. Median ties are assigned to the higher side. Support thresholds are applied before medians are calculated.

In [0]:
def segment_gender_profiles(pdf, entity_col, min_support, level_label):
    required = ["gender_clean", entity_col, "fragrance_count", "mean_votes", "mean_rating"]
    missing = [c for c in required if c not in pdf.columns]
    assert not missing, f"{level_label}: missing columns {missing}"

    data = pdf.dropna(subset=required).copy()
    for col in ["fragrance_count", "mean_votes", "mean_rating"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    data = data.dropna(subset=["fragrance_count", "mean_votes", "mean_rating"])
    data = data[data["fragrance_count"] >= min_support].copy()
    if data.empty:
        raise ValueError(f"No eligible {level_label} profiles at support >= {min_support}.")

    data["segment"] = None
    threshold_rows = []
    for gender in GENDER_ORDER:
        mask = data["gender_clean"] == gender
        part = data.loc[mask]
        if part.empty:
            continue
        attention_median = part["mean_votes"].median()
        rating_median = part["mean_rating"].median()
        high_attention = part["mean_votes"] >= attention_median
        high_rating = part["mean_rating"] >= rating_median
        data.loc[mask, "segment"] = np.select(
            [high_attention & high_rating, ~high_attention & high_rating, high_attention & ~high_rating],
            ["Standout Profiles", "Hidden Favourites", "Widely Explored"],
            default="Quiet Profiles"
        )
        threshold_rows.append({
            "classification": gender,
            "analytical_level": level_label,
            "eligible_profile_count": len(part),
            "minimum_fragrance_support": min_support,
            "attention_median": attention_median,
            "rating_median": rating_median
        })

    data["analytical_level"] = level_label
    data["segment"] = pd.Categorical(data["segment"], categories=SEGMENT_ORDER, ordered=True)
    summary = (
        data.groupby(["gender_clean", "segment"], observed=False)
        .agg(
            profile_count=(entity_col, "nunique"),
            fragrance_assignments=("fragrance_count", "sum"),
            mean_fragrances_per_profile=("fragrance_count", "mean"),
            mean_votes_per_profile=("mean_votes", "mean"),
            mean_rating_per_profile=("mean_rating", "mean"),
            median_fragrances_per_profile=("fragrance_count", "median")
        ).reset_index()
    )
    summary["analytical_level"] = level_label
    return data, pd.DataFrame(threshold_rows), summary

## 3.4 Dominant-accord segmentation by classification

The dominant accord is the accord recorded at position 1 in the original accord array. Each fragrance contributes to one dominant-accord profile, provided that a valid first accord is present. Rating and attention metrics use records with a non-null rating and non-negative vote count. Support is the number of distinct fragrances within each classification–accord group.

In [0]:
dominant_assignments_ch3 = (
    df_accords
    .select(
        "id", "gender_clean", "rating_value", "rating_count",
        F.element_at("accord_array", 1).alias("dominant_accord")
    )
    .filter(
        F.col("dominant_accord").isNotNull()
        & (F.trim(F.col("dominant_accord")) != "")
    )
)

# Each eligible fragrance must have exactly one dominant-accord assignment.
assert dominant_assignments_ch3.groupBy("id").count().filter(F.col("count") != 1).limit(1).count() == 0
assert dominant_assignments_ch3.select("id").distinct().count() == dominant_assignments_ch3.count()

dominant_profiles_spark_ch3 = (
    dominant_assignments_ch3
    .filter(F.col("rating_value").isNotNull())
    .filter(F.col("rating_count").isNotNull() & (F.col("rating_count") >= 0))
    .groupBy("gender_clean", "dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.sum("rating_count").alias("total_votes"),
        F.avg(F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)).alias("share_rated_4_5_plus")
    )
)

dominant_segmented_ch3, dominant_thresholds_ch3, dominant_summary_ch3 = segment_gender_profiles(
    dominant_profiles_spark_ch3.toPandas(), "dominant_accord",
    MIN_DOMINANT_FRAGRANCES, "Dominant accords"
)
display(dominant_thresholds_ch3)
display(dominant_summary_ch3)
display(dominant_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))

## 3.5 Position-based dyad and triad construction

Each fragrance contributes to exactly one ordered dyad (accords at positions 1 and 2) and one ordered triad (accords at positions 1, 2 and 3), where enough valid accord positions are present. Accords at positions 4 and beyond are not used to create additional combinations. The original order is preserved.

In [0]:

# Construct exactly one positional dyad and one positional triad per eligible fragrance.
# get() uses 0-based indexing and safely returns NULL for positions that do not exist.

df_positioned_accords_ch3 = (
    df_accords
    .select(
        "id",
        "gender_clean",
        "rating_value",
        "rating_count",
        F.get("accord_array", 0).alias("accord_1"),
        F.get("accord_array", 1).alias("accord_2"),
        F.get("accord_array", 2).alias("accord_3")
    )
)

def valid_accord_position(column_name):
    return (
        F.col(column_name).isNotNull()
        & (F.trim(F.col(column_name)) != "")
    )

# One dyad: position 1 + position 2.
# Fragrances with fewer than two valid positions are excluded.
df_dyad_assignments_ch3 = (
    df_positioned_accords_ch3
    .filter(
        valid_accord_position("accord_1")
        & valid_accord_position("accord_2")
    )
    .select(
        "id",
        "gender_clean",
        "rating_value",
        "rating_count",
        F.concat_ws(" + ", "accord_1", "accord_2").alias("dyad")
    )
)

# One triad: position 1 + position 2 + position 3.
# Fragrances with fewer than three valid positions are excluded.
df_triad_assignments_ch3 = (
    df_positioned_accords_ch3
    .filter(
        valid_accord_position("accord_1")
        & valid_accord_position("accord_2")
        & valid_accord_position("accord_3")
    )
    .select(
        "id",
        "gender_clean",
        "rating_value",
        "rating_count",
        F.concat_ws(
            " + ",
            "accord_1",
            "accord_2",
            "accord_3"
        ).alias("triad")
    )
)

# Validate: each eligible fragrance has exactly one assignment per level.
assert (
    df_dyad_assignments_ch3
    .groupBy("id")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count() == 0
), "A fragrance has multiple dyad assignments."

assert (
    df_triad_assignments_ch3
    .groupBy("id")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count() == 0
), "A fragrance has multiple triad assignments."

# Validate assignment totals against eligible source fragrances.
expected_dyad_count_ch3 = (
    df_positioned_accords_ch3
    .filter(
        valid_accord_position("accord_1")
        & valid_accord_position("accord_2")
    )
    .select("id")
    .distinct()
    .count()
)

expected_triad_count_ch3 = (
    df_positioned_accords_ch3
    .filter(
        valid_accord_position("accord_1")
        & valid_accord_position("accord_2")
        & valid_accord_position("accord_3")
    )
    .select("id")
    .distinct()
    .count()
)

assert df_dyad_assignments_ch3.count() == expected_dyad_count_ch3
assert df_triad_assignments_ch3.count() == expected_triad_count_ch3

# Aggregate profile-level metrics.
def aggregate_combinations(df, entity_col):
    return (
        df
        .filter(F.col("rating_value").isNotNull())
        .filter(
            F.col("rating_count").isNotNull()
            & (F.col("rating_count") >= 0)
        )
        .groupBy("gender_clean", entity_col)
        .agg(
            F.countDistinct("id").alias("fragrance_count"),
            F.avg("rating_count").alias("mean_votes"),
            F.avg("rating_value").alias("mean_rating"),
            F.sum("rating_count").alias("total_votes"),
            F.avg(
                F.when(F.col("rating_value") >= 4.5, 1.0)
                .otherwise(0.0)
            ).alias("share_rated_4_5_plus")
        )
    )

dyad_profiles_spark_ch3 = aggregate_combinations(
    df_dyad_assignments_ch3, "dyad"
)

triad_profiles_spark_ch3 = aggregate_combinations(
    df_triad_assignments_ch3, "triad"
)

print("Eligible fragrances with one dyad:", f"{expected_dyad_count_ch3:,}")
print("Eligible fragrances with one triad:", f"{expected_triad_count_ch3:,}")
print("Fragrance–dyad assignments:", f"{df_dyad_assignments_ch3.count():,}")
print("Fragrance–triad assignments:", f"{df_triad_assignments_ch3.count():,}")

display(
    dyad_profiles_spark_ch3
    .groupBy("gender_clean")
    .count()
    .orderBy("gender_clean")
)

display(
    triad_profiles_spark_ch3
    .groupBy("gender_clean")
    .count()
    .orderBy("gender_clean")
)

## 3.6 Dyad and triad segmentation

Dyads and triads are segmented independently within each classification. Support thresholds are applied before classification-specific medians are calculated.

In [0]:
dyad_segmented_ch3, dyad_thresholds_ch3, dyad_summary_ch3 = segment_gender_profiles(
    dyad_profiles_spark_ch3.toPandas(), "dyad", MIN_DYAD_FRAGRANCES, "Dyads"
)
triad_segmented_ch3, triad_thresholds_ch3, triad_summary_ch3 = segment_gender_profiles(
    triad_profiles_spark_ch3.toPandas(), "triad", MIN_TRIAD_FRAGRANCES, "Triads"
)

gender_thresholds_ch3 = pd.concat(
    [dominant_thresholds_ch3, dyad_thresholds_ch3, triad_thresholds_ch3],
    ignore_index=True
)
gender_segment_summary_ch3 = pd.concat(
    [dominant_summary_ch3, dyad_summary_ch3, triad_summary_ch3],
    ignore_index=True
)
display(gender_thresholds_ch3)
display(gender_segment_summary_ch3.sort_values(["analytical_level", "gender_clean", "segment"]))
display(dyad_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))
display(triad_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))

## 3.7 Top-three fragrance examples within each profile

For each eligible dominant accord, dyad and triad, the three fragrances with the highest vote counts are selected within the same classification, profile and segment. Third-place ties are audited. A deterministic secondary ordering retains exactly three examples when at least three candidates exist. The full ranked candidate set is retained for further review.

In [0]:

# Fragrance lookup — keep processing in Spark
fragrance_lookup_ch3 = (
    df_accords.select(
        "id",
        "gender_clean",
        "fragrance_name",
        "brand",
        F.col("rating_value").cast("double").alias("rating"),
        F.col("rating_count").cast("long").alias("rating_count")
    )
    .dropDuplicates(["id"])
)


def rank_profile_fragrances(profile_pdf, entity_col, assignment_df, level_label):

    # Convert only the relatively small segmented profile table to Spark
    profile_labels = (
        profile_pdf[["gender_clean", entity_col, "segment"]]
        .drop_duplicates()
        .copy()
    )

    profile_labels["gender_clean"] = profile_labels["gender_clean"].astype(str)
    profile_labels[entity_col] = profile_labels[entity_col].astype(str)
    profile_labels["segment"] = profile_labels["segment"].astype(str)

    profile_labels_spark = spark.createDataFrame(profile_labels)

    # Keep assignments and fragrance details in Spark
    assignments = (
        assignment_df
        .select("id", "gender_clean", entity_col)
        .dropDuplicates()
    )

    candidates = (
        assignments
        .join(
            fragrance_lookup_ch3,
            ["id", "gender_clean"],
            "left"
        )
        .join(
            profile_labels_spark,
            ["gender_clean", entity_col],
            "inner"
        )
        # Keep the example-ranking population consistent across all levels:
        # only fragrances with a valid rating and non-negative community vote count.
        .filter(
            F.col("rating").isNotNull()
            & F.col("rating_count").isNotNull()
            & (F.col("rating_count") >= 0)
        )
        .withColumn(
            "brand",
            F.coalesce(F.col("brand"), F.lit("[Brand unavailable]"))
        )
        .withColumn(
            "fragrance_name",
            F.coalesce(F.col("fragrance_name"), F.lit("[Fragrance name unavailable]"))
        )
        .withColumn("analytical_level", F.lit(level_label))
        .withColumn("classification", F.col("gender_clean"))
    )

    group_cols = ["gender_clean", "segment", entity_col]

    # Deterministic ranking within each classification–profile–segment
    ranking_window = (
        Window
        .partitionBy(*group_cols)
        .orderBy(
            F.col("rating_count").desc(),
            F.col("rating").desc_nulls_last(),
            F.col("brand").asc(),
            F.col("fragrance_name").asc(),
            F.col("id").asc()
        )
    )

    group_window = Window.partitionBy(*group_cols)

    candidates = (
        candidates
        .withColumn(
            "rank_by_votes",
            F.row_number().over(ranking_window)
        )
        .withColumn(
            "candidate_count",
            F.count("*").over(group_window)
        )
        .withColumn(
            "third_place_vote_count",
            F.max(
                F.when(
                    F.col("rank_by_votes") == 3,
                    F.col("rating_count")
                )
            ).over(group_window)
        )
        .withColumn(
            "third_place_tie_count",
            F.sum(
                F.when(
                    F.col("rating_count") == F.col("third_place_vote_count"),
                    1
                ).otherwise(0)
            ).over(group_window)
        )
    )

    return candidates


# Dominant accord assignments
dominant_assignments_ch3 = (
    df_accords
    .filter(F.col("rating_value").isNotNull())
    .filter(
        F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .select(
        "id",
        "gender_clean",
        F.element_at("accord_array", 1).alias("dominant_accord")
    )
    .dropDuplicates(["id", "gender_clean", "dominant_accord"])
)


# Rank fragrance candidates at all three analytical levels
dominant_candidates_ch3 = rank_profile_fragrances(
    dominant_segmented_ch3,
    "dominant_accord",
    dominant_assignments_ch3,
    "Dominant accords"
)

dyad_candidates_ch3 = rank_profile_fragrances(
    dyad_segmented_ch3,
    "dyad",
    df_dyad_assignments_ch3,
    "Dyads"
)

triad_candidates_ch3 = rank_profile_fragrances(
    triad_segmented_ch3,
    "triad",
    df_triad_assignments_ch3,
    "Triads"
)


# Combine all ranked candidates in Spark
all_example_candidates_ch3 = (
    dominant_candidates_ch3
    .unionByName(dyad_candidates_ch3, allowMissingColumns=True)
    .unionByName(triad_candidates_ch3, allowMissingColumns=True)
)


# Select the top three examples per profile
top_three_examples_spark_ch3 = (
    all_example_candidates_ch3
    .filter(F.col("rank_by_votes") <= 3)
)

example_columns_ch3 = [
    "analytical_level",
    "classification",
    "segment",
    "rank_by_votes",
    "dominant_accord",
    "dyad",
    "triad",
    "fragrance_name",
    "brand",
    "rating",
    "rating_count",
    "id",
    "third_place_vote_count",
    "third_place_tie_count"
]

example_columns_ch3 = [
    c for c in example_columns_ch3
    if c in top_three_examples_spark_ch3.columns
]

# Collect only the compact top-three output
top_three_examples_ch3 = (
    top_three_examples_spark_ch3
    .select(*example_columns_ch3)
    .orderBy(
        "analytical_level",
        "classification",
        "segment",
        "rank_by_votes"
    )
    .toPandas()
)

display(top_three_examples_ch3)


# Third-place tie audit — aggregate in Spark
tie_group_cols_ch3 = [
    "analytical_level",
    "classification",
    "segment",
    "dominant_accord",
    "dyad",
    "triad"
]

tie_audit_spark_ch3 = (
    all_example_candidates_ch3
    .groupBy(*tie_group_cols_ch3)
    .agg(
        F.count("id").alias("candidate_count"),
        F.first("third_place_vote_count").alias("third_place_vote_count"),
        F.first("third_place_tie_count").alias("third_place_tie_count")
    )
)

# The audit is much smaller than the full candidate dataset
tie_audit_ch3 = (
    tie_audit_spark_ch3
    .orderBy(
        "analytical_level",
        "classification",
        "segment"
    )
    .toPandas()
)

display(tie_audit_ch3)

print("Top-three examples:", f"{len(top_three_examples_ch3):,}")
print("Tie-audit profiles:", f"{len(tie_audit_ch3):,}")
print("Full ranked candidates remain in Spark.")


## 3.8 Cross-level comparison

Counts refer to eligible profiles after support thresholds. Fragrance assignments summed across profiles are not unique catalogue totals. Segment averages are unweighted means across profiles, not fragrance-weighted averages.

In [0]:
gender_level_comparison_ch3 = (
    gender_segment_summary_ch3.groupby(["gender_clean", "analytical_level"], as_index=False)
    .agg(
        eligible_profile_count=("profile_count", "sum"),
        fragrance_assignments=("fragrance_assignments", "sum"),
        mean_votes_per_profile=("mean_votes_per_profile", "mean"),
        mean_rating_per_profile=("mean_rating_per_profile", "mean")
    )
)
display(gender_level_comparison_ch3.sort_values(["gender_clean", "analytical_level"]))

gender_segment_comparison_ch3 = gender_segment_summary_ch3.pivot_table(
    index=["analytical_level", "segment"],
    columns="gender_clean",
    values="profile_count",
    aggfunc="sum"
).reindex(columns=GENDER_ORDER).reset_index()
display(gender_segment_comparison_ch3)

## 3.9 Export analytical reference tables

Complete analytical tables are exported for manuscript analysis and reproducibility. The source and prepared reference tables are not overwritten.

In [0]:
# Ensure the output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pandas DataFrames only
exports_ch3 = {
    "chapter_3_gender_distribution.csv": gender_distribution_ch3.toPandas(),
    "chapter_3_accord_prevalence.csv": accord_prevalence_ch3.toPandas(),
    "chapter_3_gender_thresholds.csv": gender_thresholds_ch3,
    "chapter_3_gender_segment_summary.csv": gender_segment_summary_ch3,
    "chapter_3_dominant_accord_profiles.csv": dominant_segmented_ch3,
    "chapter_3_dyad_profiles.csv": dyad_segmented_ch3,
    "chapter_3_triad_profiles.csv": triad_segmented_ch3,
    "chapter_3_top_three_fragrance_examples.csv": top_three_examples_ch3,
    "chapter_3_example_tie_audit.csv": tie_audit_ch3,
    "chapter_3_gender_level_comparison.csv": gender_level_comparison_ch3,
    "chapter_3_gender_segment_comparison.csv": gender_segment_comparison_ch3
}

# Export Pandas DataFrames
for filename, export_df in exports_ch3.items():
    output_path = OUTPUT_DIR / filename

    export_df.to_csv(output_path, index=False)

    assert (
        output_path.is_file() and output_path.stat().st_size > 0
    ), f"Export failed: {output_path}"

    print(
        f"Saved: {output_path} "
        f"({output_path.stat().st_size:,} bytes)"
    )

# Export the complete ranked candidates directly from Spark
# Spark creates a directory containing one or more CSV part files.
all_candidates_path = str(
    OUTPUT_DIR / "chapter_3_all_ranked_example_candidates"
)

(
    all_example_candidates_ch3
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(all_candidates_path)
)

print("Saved full ranked candidates:", all_candidates_path)

print(
    "Verified Pandas exports:", len(exports_ch3),
    "| Full ranked candidates: exported through Spark"
)

## Reproducibility notes

- Source: main.fragrance_analytics.fragrantica_perfumes.
- Clean fragrance and brand names: main.fragrance_analytics.fragrantica_prepared, joined by URL-derived fragrance_id.
- Source records are validated at one row per Fragrantica id; the reference join must preserve row count.
- Dominant accord is position 1; the dyad is positions 1+2; the triad is positions 1+2+3, preserving source order.
- Each fragrance contributes at most one profile at each analytical level, where sufficient valid accord positions exist. Positions 4+ do not generate alternative dyads or triads.
- Minimum support: 50 fragrances for dominant accords, dyads and triads.
- Medians are calculated independently by classification and analytical level; median ties are assigned to the higher side.
- Top-three examples are selected by vote count within classification–profile–segment; full ranked candidates and third-place ties are retained.
- Vote counts are platform activity, not sales, market share or unique consumers. Ratings are platform evaluations.
- Profile frequencies count distinct fragrances within each profile. Totals across different profiles at the same analytical level are not unique catalogue totals.
- This notebook produces analytical reference tables. Publication figures should follow the established Chapter 1 and Chapter 2 visual conventions.

In [0]:
# =============================================================================
# Chapter 3 — Compact analytical report (final cell)
# Reuses established notebook variables only: no recalculation, no methodology
# changes, no exports, no overwriting of analytical results. The full ranked
# candidate dataset stays in Spark; only a group-level aggregation of the same
# cardinality as the existing tie audit is collected for rank verification.
# =============================================================================
import pandas as pd
from pyspark.sql import functions as F

_required_vars = [
    "gender_distribution_ch3", "accord_prevalence_ch3",
    "dominant_thresholds_ch3", "dyad_thresholds_ch3", "triad_thresholds_ch3",
    "dominant_summary_ch3", "dyad_summary_ch3", "triad_summary_ch3",
    "dominant_segmented_ch3", "dyad_segmented_ch3", "triad_segmented_ch3",
    "top_three_examples_ch3", "tie_audit_ch3", "gender_thresholds_ch3",
    "gender_segment_summary_ch3", "gender_level_comparison_ch3",
    "gender_segment_comparison_ch3", "all_example_candidates_ch3",
]
_missing = [v for v in _required_vars if v not in globals()]
assert not _missing, f"Missing upstream results — run the preceding cells first: {_missing}"

SEGMENTS = list(SEGMENT_ORDER) if "SEGMENT_ORDER" in globals() else [
    "Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"
]
GENDERS = list(GENDER_ORDER) if "GENDER_ORDER" in globals() else None
ENTITY_COLS = ["dominant_accord", "dyad", "triad"]
LEVEL_SPECS = [
    ("Dominant accords", "dominant_accord", dominant_segmented_ch3),
    ("Dyads", "dyad", dyad_segmented_ch3),
    ("Triads", "triad", triad_segmented_ch3),
]

_report, _issues = [], []

def _hr(title):
    _report.extend(["", "=" * 74, title, "=" * 74])

def _sub(title):
    _report.extend(["", "--- " + title + " ---"])

def _check(msg, ok):
    _report.append(("  [PASS] " if ok else "  [FAIL] ") + msg)
    if not ok:
        _issues.append(msg)

_report.append("CHAPTER 3 — GENDER AND MARKET SEGMENTATION: COMPACT ANALYTICAL REPORT")
_report.append("Source: Fragrantica community data. All metrics are descriptive platform")
_report.append("measures, not market demand, sales or objective fragrance quality.")

# ---------------------------------------------------------------- 1. Catalogue
gender_dist_pdf = gender_distribution_ch3.toPandas()

# Preserve all observed classifications and order them consistently.
gender_dist_pdf["gender_clean"] = pd.Categorical(
    gender_dist_pdf["gender_clean"],
    categories=GENDERS,
    ordered=True
)
gender_dist_pdf = gender_dist_pdf.sort_values("gender_clean").reset_index(drop=True)

# Validate the underlying catalogue distribution before reporting.
assert set(gender_dist_pdf["gender_clean"].dropna()) == set(GENDERS), (
    "Catalogue distribution is missing one or more expected classifications."
)
assert gender_dist_pdf["fragrance_count"].notna().all()
assert (gender_dist_pdf["fragrance_count"] > 0).all()

total_catalogue = int(gender_dist_pdf["fragrance_count"].sum())

assert total_catalogue > 0, "Catalogue total must be greater than zero."

# Recalculate shares from the actual counts.
gender_dist_pdf["catalogue_share_pct"] = (
    gender_dist_pdf["fragrance_count"] / total_catalogue * 100
).round(2)

_hr("1. CATALOGUE DISTRIBUTION BY GENDER")
_report.append(f"Gendered fragrances in catalogue: {total_catalogue:,}")

for r in gender_dist_pdf.itertuples():
    _report.append(
        f"  {r.gender_clean:<12} {int(r.fragrance_count):>8,} fragrances  "
        f"({r.catalogue_share_pct:.2f}% of catalogue)"
    )
# ------------------------------------------------------- 2. Accord prevalence
prev_pdf = accord_prevalence_ch3.toPandas()
prev_genders = [g for g in (GENDERS or sorted(prev_pdf["gender_clean"].unique()))
                if g in set(prev_pdf["gender_clean"])]
piv = prev_pdf.pivot_table(
    index="accord", columns="gender_clean", values="prevalence_pct"
).reindex(columns=prev_genders).round(2)
piv["spread_pp"] = (piv.max(axis=1) - piv.min(axis=1)).round(2)
_hr("2. ACCORD PREVALENCE BY GENDER (share of fragrances within each classification)")
_report.append(f"Distinct accords: {piv.shape[0]:,} | classifications: {', '.join(prev_genders)}")
for g in prev_genders:
    top = prev_pdf[prev_pdf["gender_clean"] == g].nlargest(5, "prevalence_pct")
    _report.append(f"  Top accords — {g}: " + "; ".join(
        f"{r.accord} ({r.prevalence_pct:.1f}%)" for r in top.itertuples()))
_sub("Most notable prevalence differences (percentage-point spread between the "
      "highest and lowest classification)")
for accord, r in piv.sort_values("spread_pp", ascending=False).head(10).iterrows():
    _report.append(f"  {str(accord)[:26]:<26} spread {r['spread_pp']:>6.2f} pp  " +
                   ", ".join(f"{g}: {r[g]:.2f}%" for g in prev_genders))

# ----------------------------------------------- 3-5. Segmentation by level
def _segmentation_block(number, title, thresholds_df, summary_df, segmented_df):
    _hr(f"{number}. {title}")
    thr = thresholds_df.copy()
    support_vals = sorted({int(v) for v in thr["minimum_fragrance_support"]})
    _report.append("Support threshold applied before medians: " +
                   ", ".join(f"{v}" for v in support_vals) +
                   " fragrances per classification–profile group")
    _report.append(f"Eligible profiles after support: {int(thr['eligible_profile_count'].sum()):,} "
                   f"across {len(thr)} classifications")
    for _, t in thr.iterrows():
        _report.append(
            f"  {t['classification']:<12} eligible profiles {int(t['eligible_profile_count']):>5} | "
            f"attention median (mean votes) {t['attention_median']:.2f} | "
            f"rating median {t['rating_median']:.2f}")
    _sub("Segments (unweighted per-profile means; assignments are not unique fragrances; "
          "medians per classification and level, ties to the higher side)")
    s = summary_df.copy()
    order_g = [g for g in (GENDERS or []) if g in set(s["gender_clean"])]
    if order_g:
        s["_g"] = pd.Categorical(s["gender_clean"], categories=order_g, ordered=True)
        s = s.sort_values(["_g", "segment"])
    else:
        s = s.sort_values(["gender_clean", "segment"])
    for _, r in s.iterrows():
        _report.append(
            f"  {r['gender_clean']:<12} {r['segment']:<18} profiles {int(r['profile_count']):>4} | "
            f"assignments {int(r['fragrance_assignments']):>7,} | "
            f"mean votes {r['mean_votes_per_profile']:>7.1f} | "
            f"mean rating {r['mean_rating_per_profile']:.2f} | "
            f"median fragrances {r['median_fragrances_per_profile']:.1f}")

_segmentation_block(3, "DOMINANT-ACCORD SEGMENTATION",
                    dominant_thresholds_ch3, dominant_summary_ch3, dominant_segmented_ch3)
_segmentation_block(4, "DYAD SEGMENTATION",
                    dyad_thresholds_ch3, dyad_summary_ch3, dyad_segmented_ch3)
_segmentation_block(5, "TRIAD SEGMENTATION",
                    triad_thresholds_ch3, triad_summary_ch3, triad_segmented_ch3)

# --------------------------------------------- 6. Top-three example profiles
examples = top_three_examples_ch3.copy()
_parts = []
for level, ecol, segmented_df in LEVEL_SPECS:
    lvl = examples[examples["analytical_level"] == level].copy()
    if lvl.empty:
        continue
    lvl["profile"] = lvl[ecol]
    freq = (segmented_df[["gender_clean", ecol, "fragrance_count"]]
            .drop_duplicates()
            .rename(columns={"gender_clean": "classification", ecol: "profile",
                             "fragrance_count": "profile_frequency"}))
    _parts.append(lvl.merge(freq, on=["classification", "profile"], how="left"))
assert _parts, "No top-three examples found in any analytical level."
examples = pd.concat(_parts, ignore_index=True)

_hr("6. TOP-THREE FRAGRANCE EXAMPLES PER ELIGIBLE PROFILE")
_report.append("Selection: highest community vote counts within classification–profile–segment;")
_report.append("ties broken deterministically (rating desc, brand, name, id). 'Profile frequency'")
_report.append("= distinct fragrances in that profile within the classification. Previews below;")
_report.append("the complete table remains in `examples` for review.")
_grp_cols_ex = ["analytical_level", "classification", "segment", "profile"]
for level, _, _ in LEVEL_SPECS:
    lvl = examples[examples["analytical_level"] == level]
    n_groups = lvl.groupby(_grp_cols_ex).ngroups
    _report.append(f"  {level}: {n_groups:,} profile groups, {len(lvl):,} example rows")
for level, _, _ in LEVEL_SPECS:
    lvl = examples[examples["analytical_level"] == level]
    _sub(f"{level} — highest-voted examples (preview, up to 12 of {len(lvl):,} rows)")
    for r in lvl.nlargest(12, "rating_count").itertuples():
        rating = f"{r.rating:.2f}" if pd.notna(r.rating) else "n/a"
        freq = f"{int(r.profile_frequency):,}" if pd.notna(r.profile_frequency) else "n/a"
        _report.append(
            f"  {r.classification:<11} | {r.segment:<18} | {str(r.profile)[:34]:<34} | "
            f"#{int(r.rank_by_votes)} | {str(r.fragrance_name)[:28]:<28} | "
            f"{str(r.brand)[:20]:<20} | rating {rating} | "
            f"votes {int(r.rating_count):>7,} | profile frequency {freq}")

# ------------------------------------------------------ 7. Rank verification
_hr("7. RANK VERIFICATION (top-three ordered by community vote count within "
    "classification–profile–segment)")
_grp = examples.groupby(_grp_cols_ex, sort=False)
_bad_order = []
for key, d in _grp:
    v = list(d.sort_values("rank_by_votes")["rating_count"])
    if any(v[i] < v[i + 1] for i in range(len(v) - 1)):
        _bad_order.append(key)
_check(f"Vote counts non-increasing across ranks 1–3 in all {_grp.ngroups:,} groups",
       not _bad_order)
for k in _bad_order[:5]:
    _report.append(f"    offending group: {k}")

# Verify against the full ranked candidates in Spark (group-level only — one
# row per eligible profile group, same cardinality as the existing tie audit).
_spark_group_cols = ["analytical_level", "classification", "segment"] + ENTITY_COLS
spark_group_max = (
    all_example_candidates_ch3
    .groupBy(*_spark_group_cols)
    .agg(F.max("rating_count").alias("spark_group_max_votes"))
    .toPandas()
)
for c in ENTITY_COLS:
    spark_group_max[c] = spark_group_max[c].fillna("__na__")
rank1 = examples[examples["rank_by_votes"] == 1].copy()
for c in ENTITY_COLS:
    rank1[c] = rank1[c].fillna("__na__")
merged = rank1[_spark_group_cols + ["rating_count"]].merge(
    spark_group_max, on=_spark_group_cols, how="inner")
_check(f"Every rank-1 example matched its full-candidate group in Spark "
       f"({len(merged):,} matched, {len(rank1) - len(merged):,} unmatched)",
       len(merged) == len(rank1))
_mismatch = merged[merged["rating_count"] != merged["spark_group_max_votes"]]
_check("Rank-1 example vote counts equal the full-candidate group maximum",
       len(_mismatch) == 0)

# ------------------------------------------------------- 8. Third-place ties
tie = tie_audit_ch3.copy()
tie["third_place_tie_count"] = tie["third_place_tie_count"].fillna(0)
ties = tie[tie["third_place_tie_count"] > 1]
undetermined = tie[tie["third_place_tie_count"] > 3]
short_groups = tie[tie["candidate_count"] < 3]
_hr("8. THIRD-PLACE TIE AUDIT")
_report.append(f"Audited profile groups: {len(tie):,}")
_report.append(f"  Groups with fewer than three ranked candidates: {len(short_groups):,}")
_report.append(f"  Groups with a third-place vote tie (tie count > 1): {len(ties):,}")
_report.append(f"  Groups where more than three candidates tie at the third-place vote "
               f"count (deterministic secondary ordering selected the shown examples): "
               f"{len(undetermined):,}")
if len(ties):
    _sub("Largest third-place ties (up to 10)")
    for r in ties.nlargest(10, "third_place_tie_count").itertuples():
        _report.append(
            f"  {r.analytical_level} | {r.classification} | {r.segment} | "
            f"third-place votes {int(r.third_place_vote_count):,} | "
            f"tied candidates at that count: {int(r.third_place_tie_count)}")

def _group_keys(df):
    ent = df[ENTITY_COLS].copy()
    for c in ENTITY_COLS:
        ent[c] = ent[c].fillna("__na__")
    return set(zip(df["analytical_level"], df["classification"], df["segment"],
                   list(ent.itertuples(index=False, name=None))))

_no_examples = _group_keys(tie) - _group_keys(examples)
_check(f"Every audited profile group has at least one top-three example "
       f"(groups without examples: {len(_no_examples):,})", not _no_examples)
_expected_rows = int(tie["candidate_count"].clip(upper=3).sum())
_check(f"Top-three row count equals sum of min(3, candidate_count) over audited "
       f"groups (expected {_expected_rows:,}, actual {len(examples):,})",
       _expected_rows == len(examples))

# ------------------------------------------------- 9. Cross-level comparison
_hr("9. GENDER-LEVEL AND GENDER-SEGMENT COMPARISONS")
_report.append("Counts are eligible profiles after support. Fragrance assignments summed "
               "across profiles are not unique catalogue totals.")
for _, r in gender_level_comparison_ch3.iterrows():
    _report.append(
        f"  {r['gender_clean']:<12} {r['analytical_level']:<17} "
        f"profiles {int(r['eligible_profile_count']):>5} | "
        f"assignments {int(r['fragrance_assignments']):>7,} | "
        f"mean votes/profile {r['mean_votes_per_profile']:>7.1f} | "
        f"mean rating/profile {r['mean_rating_per_profile']:.2f}")
_sub("Profile counts by analytical level and segment (pivot)")
_pivot_cols = [c for c in gender_segment_comparison_ch3.columns
               if c not in ("analytical_level", "segment")]
for _, r in gender_segment_comparison_ch3.iterrows():
    vals = " | ".join(
        f"{g}: {int(r[g]) if pd.notna(r[g]) else 0}" for g in _pivot_cols)
    _report.append(f"  {r['analytical_level']:<17} {r['segment']:<18} {vals}")

# ------------------------------------------- 10. Validation and data quality
_hr("10. VALIDATION AND DATA-QUALITY CHECKS")
for label, segmented_df, thresholds_df in [
    ("Dominant accords", dominant_segmented_ch3, dominant_thresholds_ch3),
    ("Dyads", dyad_segmented_ch3, dyad_thresholds_ch3),
    ("Triads", triad_segmented_ch3, triad_thresholds_ch3),
]:
    required = int(thresholds_df["minimum_fragrance_support"].iloc[0])
    actual_min = int(segmented_df["fragrance_count"].min())
    _check(f"{label}: minimum observed profile support {actual_min:,} >= "
           f"required {required:,}", actual_min >= required)
    thr_total = int(thresholds_df["eligible_profile_count"].sum())
    _check(f"{label}: segmented profile rows {len(segmented_df):,} equal "
           f"threshold-eligible profiles {thr_total:,}",
           len(segmented_df) == thr_total)
for label in ["Dominant accords", "Dyads", "Triads"]:
    comp_total = int(gender_level_comparison_ch3.loc[
        gender_level_comparison_ch3["analytical_level"] == label,
        "eligible_profile_count"].sum())
    thr_total = int(gender_thresholds_ch3.loc[
        gender_thresholds_ch3["analytical_level"] == label,
        "eligible_profile_count"].sum())
    _check(f"{label}: comparison-table eligible profiles {comp_total:,} match "
           f"thresholds {thr_total:,}", comp_total == thr_total)

_bad_labels = sorted(set(examples["segment"].astype(str)) - set(SEGMENTS))
_check(f"All example segments use the established labels {SEGMENTS}", not _bad_labels)
_null_rating = int(examples["rating"].isna().sum())
_null_votes = int(examples["rating_count"].isna().sum())
_null_freq = int(examples["profile_frequency"].isna().sum())
_ph_name = int((examples["fragrance_name"] == "[Fragrance name unavailable]").sum())
_ph_brand = int((examples["brand"] == "[Brand unavailable]").sum())
_report.append(f"  Top-three examples — null ratings: {_null_rating:,}; "
               f"null vote counts: {_null_votes:,}; missing profile frequency: {_null_freq:,}")
_report.append(f"  Top-three examples — placeholder fragrance names: {_ph_name:,}; "
               f"placeholder brands: {_ph_brand:,}")
if _null_freq:
    _issues.append(f"{_null_freq} top-three examples missing profile frequency")
if _null_rating:
    _issues.append(f"{_null_rating} top-three examples missing ratings")
_share_sum = float(gender_dist_pdf["catalogue_share_pct"].sum())
_check(f"Catalogue share percentages sum to {_share_sum:.2f} (within rounding of 100)",
       abs(_share_sum - 100.0) <= 0.05)

_hr("VALIDATION SUMMARY")
if _issues:
    _report.append(f"{len(_issues)} issue(s) detected:")
    for msg in _issues:
        _report.append(f"  - {msg}")
else:
    _report.append("All validation checks passed.")

_hr("INTERPRETATION CAVEATS")
_report.append("- Catalogue prevalence, platform vote counts and user ratings reflect")
_report.append("  Fragrantica community activity only. They are NOT measures of market")
_report.append("  demand, sales, market share, consumer reach or objective fragrance quality.")
_report.append("- Fragrance assignments summed across profiles are not unique catalogue totals;")
_report.append("  segment averages are unweighted means across profiles.")
_report.append("- Medians are calculated independently per classification and analytical level;")
_report.append("  median ties are assigned to the higher side. Support thresholds are applied")
_report.append("  before medians.")
_report.append("- Results are descriptive and platform-specific. No causal claims are made,")
_report.append("  and findings are not generalised to all fragrance consumers.")

chapter3_report_text = "\n".join(_report)
print(chapter3_report_text)
print(f"\nReport stored in `chapter3_report_text` ({len(_report):,} lines) for copying.")
print("Full ranked candidate dataset remains in Spark; nothing was exported or overwritten.")

In [0]:

# =============================================================================
# DYAD ANALYSIS — COMPACT REPORT AND BUBBLE-CHART DATA
# Uses existing Notebook 18 results. No exports or methodology changes.
# =============================================================================

import pandas as pd

_REQUIRED = [
    "dyad_segmented_ch3",
    "dyad_thresholds_ch3",
    "dyad_summary_ch3",
    "top_three_examples_ch3",
]

_missing = [v for v in _REQUIRED if v not in globals()]
assert not _missing, f"Run the preceding Notebook 18 cells first: {_missing}"

GENDER_ORDER = ["Men", "Women", "Unisex"]
SEGMENT_ORDER = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

# ---------------------------------------------------------------------------
# 1. Individual dyad profiles — exact input for the three bubble charts
# ---------------------------------------------------------------------------

dyad_chart_data_ch3 = dyad_segmented_ch3[
    [
        "gender_clean",
        "dyad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "segment",
    ]
].copy()

dyad_chart_data_ch3 = dyad_chart_data_ch3.rename(columns={
    "gender_clean": "classification",
    "fragrance_count": "fragrance_count",
    "mean_votes": "mean_votes_per_fragrance",
    "mean_rating": "mean_rating",
})

dyad_chart_data_ch3["classification"] = pd.Categorical(
    dyad_chart_data_ch3["classification"],
    categories=GENDER_ORDER,
    ordered=True
)

dyad_chart_data_ch3["segment"] = pd.Categorical(
    dyad_chart_data_ch3["segment"],
    categories=SEGMENT_ORDER,
    ordered=True
)

dyad_chart_data_ch3 = (
    dyad_chart_data_ch3
    .sort_values(
        ["classification", "segment", "mean_votes_per_fragrance"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 2. Classification-specific median thresholds
# ---------------------------------------------------------------------------

dyad_threshold_report_ch3 = (
    dyad_thresholds_ch3
    .sort_values("classification")
    .reset_index(drop=True)
    .copy()
)

# ---------------------------------------------------------------------------
# 3. Segment-level summary
# ---------------------------------------------------------------------------

dyad_segment_report_ch3 = dyad_summary_ch3.copy()

dyad_segment_report_ch3["gender_clean"] = pd.Categorical(
    dyad_segment_report_ch3["gender_clean"],
    categories=GENDER_ORDER,
    ordered=True
)

dyad_segment_report_ch3["segment"] = pd.Categorical(
    dyad_segment_report_ch3["segment"],
    categories=SEGMENT_ORDER,
    ordered=True
)

dyad_segment_report_ch3 = (
    dyad_segment_report_ch3
    .sort_values(["gender_clean", "segment"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 4. Top-three fragrance examples for dyads
# ---------------------------------------------------------------------------

dyad_examples_report_ch3 = (
    top_three_examples_ch3[
        top_three_examples_ch3["analytical_level"] == "Dyads"
    ]
    .copy()
)

dyad_frequency_lookup = (
    dyad_segmented_ch3[
        ["gender_clean", "dyad", "fragrance_count"]
    ]
    .drop_duplicates()
    .rename(columns={
        "gender_clean": "classification",
        "fragrance_count": "profile_frequency"
    })
)

dyad_examples_report_ch3 = dyad_examples_report_ch3.merge(
    dyad_frequency_lookup,
    on=["classification", "dyad"],
    how="left"
)

dyad_example_columns = [
    "classification",
    "segment",
    "dyad",
    "rank_by_votes",
    "fragrance_name",
    "brand",
    "rating",
    "rating_count",
    "profile_frequency",
]

dyad_examples_report_ch3 = (
    dyad_examples_report_ch3[dyad_example_columns]
    .sort_values(
        ["classification", "segment", "dyad", "rank_by_votes"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 5. Validation checks
# ---------------------------------------------------------------------------

dyad_checks_ch3 = []

def dyad_check(label, condition):
    dyad_checks_ch3.append({
        "check": label,
        "status": "PASS" if condition else "FAIL"
    })
    assert condition, label

dyad_check(
    "Expected eligible dyad profile count",
    len(dyad_chart_data_ch3) ==
    int(dyad_threshold_report_ch3["eligible_profile_count"].sum())
)

dyad_check(
    "Minimum profile support is at least 50 fragrances",
    dyad_chart_data_ch3["fragrance_count"].min() >= MIN_DYAD_FRAGRANCES
)

dyad_check(
    "All three classifications are represented",
    set(dyad_chart_data_ch3["classification"].dropna().astype(str))
    == set(GENDER_ORDER)
)

dyad_check(
    "All four established segments are represented",
    set(dyad_chart_data_ch3["segment"].dropna().astype(str))
    == set(SEGMENT_ORDER)
)

dyad_check(
    "No missing bubble-chart metrics",
    dyad_chart_data_ch3[
        [
            "fragrance_count",
            "mean_votes_per_fragrance",
            "mean_rating"
        ]
    ].notna().all().all()
)

dyad_check(
    "All top-three examples have profile frequency",
    dyad_examples_report_ch3["profile_frequency"].notna().all()
)

dyad_check(
    "Top-three example rows match the expected count",
    len(dyad_examples_report_ch3) ==
    sum(min(3, n) for n in
        dyad_examples_report_ch3.groupby(
            ["classification", "dyad", "segment"]
        ).size())
)

# ---------------------------------------------------------------------------
# 6. Compact printed report
# ---------------------------------------------------------------------------

_report = []

_report.append("DYAD ANALYSIS — COMPACT REPORT")
_report.append("=" * 72)
_report.append(
    f"Eligible dyad profiles: {len(dyad_chart_data_ch3):,}"
)
_report.append(
    f"Minimum fragrance support: {MIN_DYAD_FRAGRANCES}"
)
_report.append(
    "Dyad construction: accord positions 1 + 2; original order preserved."
)

_report.append("")
_report.append("CLASSIFICATION-SPECIFIC MEDIANS")
_report.append("-" * 72)

for r in dyad_threshold_report_ch3.itertuples():
    _report.append(
        f"{r.classification:<10} "
        f"profiles: {int(r.eligible_profile_count):>4} | "
        f"attention median: {r.attention_median:>8.2f} | "
        f"rating median: {r.rating_median:.4f}"
    )

_report.append("")
_report.append("SEGMENT SUMMARY")
_report.append("-" * 72)

for r in dyad_segment_report_ch3.itertuples():
    _report.append(
        f"{str(r.gender_clean):<10} "
        f"{str(r.segment):<20} "
        f"profiles: {int(r.profile_count):>3} | "
        f"assignments: {int(r.fragrance_assignments):>6,} | "
        f"mean votes/profile: {r.mean_votes_per_profile:>7.1f} | "
        f"mean rating/profile: {r.mean_rating_per_profile:.3f} | "
        f"median frequency: {r.median_fragrances_per_profile:.1f}"
    )

_report.append("")
_report.append("BUBBLE-CHART DATA")
_report.append("-" * 72)
_report.append(
    "One row per eligible dyad profile; use these fields for the charts:"
)
_report.append(
    "classification | dyad | fragrance_count | "
    "mean_votes_per_fragrance | mean_rating | segment"
)
_report.append(
    f"Total chart-data rows: {len(dyad_chart_data_ch3):,}"
)

_report.append("")
_report.append("TOP-THREE EXAMPLES")
_report.append("-" * 72)
_report.append(
    f"Dyad profile groups: "
    f"{dyad_examples_report_ch3['dyad'].nunique():,} "
    f"(within classification and segment)"
)
_report.append(
    f"Example rows: {len(dyad_examples_report_ch3):,}"
)

_report.append("")
_report.append("VALIDATION")
_report.append("-" * 72)

for r in dyad_checks_ch3:
    _report.append(f"[{r['status']}] {r['check']}")

dyad_report_text_ch3 = "\n".join(_report)

print(dyad_report_text_ch3)

# ---------------------------------------------------------------------------
# 7. Databricks table displays — no files created
# ---------------------------------------------------------------------------

print("\n1. MEDIAN THRESHOLDS")
display(dyad_threshold_report_ch3)

print("\n2. SEGMENT SUMMARY")
display(dyad_segment_report_ch3)

print("\n3. INDIVIDUAL DYAD PROFILES — BUBBLE-CHART INPUT")
display(dyad_chart_data_ch3)

print("\n4. TOP-THREE DYAD FRAGRANCE EXAMPLES")
display(dyad_examples_report_ch3)

print("\n5. VALIDATION RESULTS")
display(pd.DataFrame(dyad_checks_ch3))


In [0]:
# ---------------------------------------------------------------------------
# 5. SELECTED DYAD EXAMPLES — TOP 3 DYADS PER SEGMENT
# ---------------------------------------------------------------------------

dyad_selection_ch3 = dyad_chart_data_ch3.copy()

dyad_selection_ch3["total_votes"] = (
    dyad_selection_ch3["mean_votes_per_fragrance"]
    * dyad_selection_ch3["fragrance_count"]
)

dyad_selection_ch3 = (
    dyad_selection_ch3
    .sort_values(
        [
            "classification",
            "segment",
            "total_votes",
            "dyad"
        ],
        ascending=[True, True, False, True]
    )
    .groupby(
        ["classification", "segment"],
        observed=True,
        sort=False
    )
    .head(3)
    .reset_index(drop=True)
)

selected_dyad_examples_ch3 = (
    dyad_examples_report_ch3.merge(
        dyad_selection_ch3[
            ["classification", "segment", "dyad", "total_votes"]
        ],
        on=["classification", "segment", "dyad"],
        how="inner"
    )
    .sort_values(
        [
            "classification",
            "segment",
            "total_votes",
            "rank_by_votes"
        ],
        ascending=[True, True, False, True]
    )
    .reset_index(drop=True)
)

print("SELECTED DYADS AND FRAGRANCE EXAMPLES")
display(selected_dyad_examples_ch3)

In [0]:

# =============================================================================
# DYAD CROSS-CLASSIFICATION ANALYSIS
# Identifies dyad combinations appearing in multiple gender classifications.
# Uses existing dyad bubble-chart data. No changes to source data.
# =============================================================================

# ---------------------------------------------------------------------------
# 1. Prepare comparison data
# ---------------------------------------------------------------------------

shared_dyad_source_ch3 = dyad_chart_data_ch3.copy()

# Treat accord order as interchangeable for cross-classification matching.
# Preserve the original dyad label in a separate column.

shared_dyad_source_ch3["profile_key"] = (
    shared_dyad_source_ch3["dyad"]
    .apply(lambda x: " + ".join(sorted(str(x).split(" + "))))
)

shared_dyad_source_ch3["classification"] = (
    shared_dyad_source_ch3["classification"].astype(str)
)

shared_dyad_source_ch3["segment"] = (
    shared_dyad_source_ch3["segment"].astype(str)
)

# ---------------------------------------------------------------------------
# 2. Identify combinations appearing in at least two classifications
# ---------------------------------------------------------------------------

dyad_classification_counts_ch3 = (
    shared_dyad_source_ch3
    .groupby("profile_key")["classification"]
    .nunique()
)

shared_dyad_keys_ch3 = dyad_classification_counts_ch3[
    dyad_classification_counts_ch3 >= 2
].index

shared_dyad_profiles_ch3 = (
    shared_dyad_source_ch3[
        shared_dyad_source_ch3["profile_key"].isin(shared_dyad_keys_ch3)
    ]
    .copy()
    .sort_values(
        ["profile_key", "classification", "segment"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 3. Summarise each shared combination
# ---------------------------------------------------------------------------

shared_dyad_summary_ch3 = (
    shared_dyad_profiles_ch3
    .groupby("profile_key")
    .agg(
        classification_count=("classification", "nunique"),
        classifications=(
            "classification",
            lambda x: ", ".join(sorted(set(x)))
        ),
        original_dyad_labels=(
            "dyad",
            lambda x: " | ".join(dict.fromkeys(x))
        ),
        segments=(
            "segment",
            lambda x: " | ".join(dict.fromkeys(x))
        ),
        segment_count=("segment", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["classification_count", "profile_key"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 4. Classification coverage summary
# ---------------------------------------------------------------------------

shared_dyad_coverage_ch3 = (
    shared_dyad_summary_ch3["classification_count"]
    .value_counts()
    .sort_index()
    .rename_axis("gender_categories")
    .reset_index(name="shared_profile_count")
)

shared_dyad_coverage_ch3["gender_categories"] = (
    shared_dyad_coverage_ch3["gender_categories"]
    .map({
        2: "Appearing in 2 classifications",
        3: "Appearing in all 3 classifications"
    })
)

# ---------------------------------------------------------------------------
# 5. Identify combinations assigned to different segments
# ---------------------------------------------------------------------------

shared_dyad_segment_differences_ch3 = (
    shared_dyad_summary_ch3[
        shared_dyad_summary_ch3["segment_count"] > 1
    ]
    .copy()
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 6. Compact printed report
# ---------------------------------------------------------------------------

_report = []

_report.append("DYAD CROSS-CLASSIFICATION ANALYSIS")
_report.append("=" * 72)

_report.append(
    f"Total eligible dyad profiles: {len(dyad_chart_data_ch3):,}"
)

_report.append(
    f"Shared dyad combinations: {len(shared_dyad_summary_ch3):,}"
)

_report.append(
    f"Combinations appearing in all three classifications: "
    f"{(shared_dyad_summary_ch3['classification_count'] == 3).sum():,}"
)

_report.append(
    f"Shared combinations with different segment assignments: "
    f"{len(shared_dyad_segment_differences_ch3):,}"
)

_report.append("")
_report.append("CLASSIFICATION COVERAGE")
_report.append("-" * 72)

for r in shared_dyad_coverage_ch3.itertuples():
    _report.append(
        f"{r.gender_categories:<38} "
        f"{r.shared_profile_count:>4}"
    )

_report.append("")
_report.append("SEGMENT DIFFERENCES")
_report.append("-" * 72)

for r in shared_dyad_segment_differences_ch3.itertuples():
    _report.append(
        f"{r.profile_key:<35} "
        f"{r.classifications:<20} "
        f"{r.segments}"
    )

print("\n".join(_report))

# ---------------------------------------------------------------------------
# 7. Databricks table displays — no files created
# ---------------------------------------------------------------------------

print("\n1. ALL SHARED DYAD COMBINATIONS")
display(shared_dyad_summary_ch3)

print("\n2. DETAILED CLASSIFICATION COMPARISON")
display(shared_dyad_profiles_ch3)

print("\n3. SHARED PROFILE COVERAGE")
display(shared_dyad_coverage_ch3)

print("\n4. SHARED PROFILES WITH DIFFERENT SEGMENT ASSIGNMENTS")
display(shared_dyad_segment_differences_ch3)


In [0]:

# =============================================================================
# TRIAD ANALYSIS — COMPACT REPORT AND BUBBLE-CHART DATA
# Uses existing Notebook 18 results. No exports or methodology changes.
# =============================================================================

import pandas as pd

_REQUIRED = [
    "triad_segmented_ch3",
    "triad_thresholds_ch3",
    "triad_summary_ch3",
    "top_three_examples_ch3",
]

_missing = [v for v in _REQUIRED if v not in globals()]
assert not _missing, f"Run the preceding Notebook 18 cells first: {_missing}"

GENDER_ORDER = ["Men", "Women", "Unisex"]
SEGMENT_ORDER = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

# ---------------------------------------------------------------------------
# 1. Individual triad profiles — exact input for the three bubble charts
# ---------------------------------------------------------------------------

triad_chart_data_ch3 = triad_segmented_ch3[
    [
        "gender_clean",
        "triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "segment",
    ]
].copy()

triad_chart_data_ch3 = triad_chart_data_ch3.rename(columns={
    "gender_clean": "classification",
    "fragrance_count": "fragrance_count",
    "mean_votes": "mean_votes_per_fragrance",
    "mean_rating": "mean_rating",
})

triad_chart_data_ch3["classification"] = pd.Categorical(
    triad_chart_data_ch3["classification"],
    categories=GENDER_ORDER,
    ordered=True
)

triad_chart_data_ch3["segment"] = pd.Categorical(
    triad_chart_data_ch3["segment"],
    categories=SEGMENT_ORDER,
    ordered=True
)

triad_chart_data_ch3 = (
    triad_chart_data_ch3
    .sort_values(
        ["classification", "segment", "mean_votes_per_fragrance"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 2. Classification-specific median thresholds
# ---------------------------------------------------------------------------

triad_threshold_report_ch3 = (
    triad_thresholds_ch3
    .sort_values("classification")
    .reset_index(drop=True)
    .copy()
)

# ---------------------------------------------------------------------------
# 3. Segment-level summary
# ---------------------------------------------------------------------------

triad_segment_report_ch3 = triad_summary_ch3.copy()

triad_segment_report_ch3["gender_clean"] = pd.Categorical(
    triad_segment_report_ch3["gender_clean"],
    categories=GENDER_ORDER,
    ordered=True
)

triad_segment_report_ch3["segment"] = pd.Categorical(
    triad_segment_report_ch3["segment"],
    categories=SEGMENT_ORDER,
    ordered=True
)

triad_segment_report_ch3 = (
    triad_segment_report_ch3
    .sort_values(["gender_clean", "segment"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 4. Top-three fragrance examples for triads
# ---------------------------------------------------------------------------

triad_examples_report_ch3 = (
    top_three_examples_ch3[
        top_three_examples_ch3["analytical_level"] == "Triads"
    ]
    .copy()
)

triad_frequency_lookup = (
    triad_segmented_ch3[
        ["gender_clean", "triad", "fragrance_count"]
    ]
    .drop_duplicates()
    .rename(columns={
        "gender_clean": "classification",
        "fragrance_count": "profile_frequency"
    })
)

triad_examples_report_ch3 = triad_examples_report_ch3.merge(
    triad_frequency_lookup,
    on=["classification", "triad"],
    how="left"
)

triad_example_columns = [
    "classification",
    "segment",
    "triad",
    "rank_by_votes",
    "fragrance_name",
    "brand",
    "rating",
    "rating_count",
    "profile_frequency",
]

triad_examples_report_ch3 = (
    triad_examples_report_ch3[triad_example_columns]
    .sort_values(
        ["classification", "segment", "triad", "rank_by_votes"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 5. Validation checks
# ---------------------------------------------------------------------------

triad_checks_ch3 = []

def triad_check(label, condition):
    triad_checks_ch3.append({
        "check": label,
        "status": "PASS" if condition else "FAIL"
    })
    assert condition, label

triad_check(
    "Expected eligible triad profile count",
    len(triad_chart_data_ch3) ==
    int(triad_threshold_report_ch3["eligible_profile_count"].sum())
)

triad_check(
    "Minimum profile support is at least 50 fragrances",
    triad_chart_data_ch3["fragrance_count"].min() >= MIN_TRIAD_FRAGRANCES
)

triad_check(
    "All three classifications are represented",
    set(triad_chart_data_ch3["classification"].dropna().astype(str))
    == set(GENDER_ORDER)
)

triad_check(
    "All four established segments are represented",
    set(triad_chart_data_ch3["segment"].dropna().astype(str))
    == set(SEGMENT_ORDER)
)

triad_check(
    "No missing bubble-chart metrics",
    triad_chart_data_ch3[
        [
            "fragrance_count",
            "mean_votes_per_fragrance",
            "mean_rating"
        ]
    ].notna().all().all()
)

triad_check(
    "All top-three examples have profile frequency",
    triad_examples_report_ch3["profile_frequency"].notna().all()
)

triad_check(
    "Top-three example rows match the expected count",
    len(triad_examples_report_ch3) ==
    sum(min(3, n) for n in
        triad_examples_report_ch3.groupby(
            ["classification", "triad", "segment"]
        ).size())
)

# ---------------------------------------------------------------------------
# 6. Compact printed report
# ---------------------------------------------------------------------------

_report = []

_report.append("TRIAD ANALYSIS — COMPACT REPORT")
_report.append("=" * 72)
_report.append(
    f"Eligible triad profiles: {len(triad_chart_data_ch3):,}"
)
_report.append(
    f"Minimum fragrance support: {MIN_TRIAD_FRAGRANCES}"
)
_report.append(
    "Triad construction: accord positions 1 + 2 + 3; original order preserved."
)

_report.append("")
_report.append("CLASSIFICATION-SPECIFIC MEDIANS")
_report.append("-" * 72)

for r in triad_threshold_report_ch3.itertuples():
    _report.append(
        f"{r.classification:<10} "
        f"profiles: {int(r.eligible_profile_count):>4} | "
        f"attention median: {r.attention_median:>8.2f} | "
        f"rating median: {r.rating_median:.4f}"
    )

_report.append("")
_report.append("SEGMENT SUMMARY")
_report.append("-" * 72)

for r in triad_segment_report_ch3.itertuples():
    _report.append(
        f"{str(r.gender_clean):<10} "
        f"{str(r.segment):<20} "
        f"profiles: {int(r.profile_count):>3} | "
        f"assignments: {int(r.fragrance_assignments):>6,} | "
        f"mean votes/profile: {r.mean_votes_per_profile:>7.1f} | "
        f"mean rating/profile: {r.mean_rating_per_profile:.3f} | "
        f"median frequency: {r.median_fragrances_per_profile:.1f}"
    )

_report.append("")
_report.append("BUBBLE-CHART DATA")
_report.append("-" * 72)
_report.append(
    "One row per eligible triad profile; use these fields for the charts:"
)
_report.append(
    "classification | triad | fragrance_count | "
    "mean_votes_per_fragrance | mean_rating | segment"
)
_report.append(
    f"Total chart-data rows: {len(triad_chart_data_ch3):,}"
)

_report.append("")
_report.append("TOP-THREE EXAMPLES")
_report.append("-" * 72)
_report.append(
    f"Triad profile groups: "
    f"{triad_examples_report_ch3['triad'].nunique():,} "
    "(within classification and segment)"
)
_report.append(
    f"Example rows: {len(triad_examples_report_ch3):,}"
)

_report.append("")
_report.append("VALIDATION")
_report.append("-" * 72)

for r in triad_checks_ch3:
    _report.append(f"[{r['status']}] {r['check']}")

triad_report_text_ch3 = "\n".join(_report)

print(triad_report_text_ch3)

# ---------------------------------------------------------------------------
# 7. Databricks table displays — no files created
# ---------------------------------------------------------------------------

print("\n1. MEDIAN THRESHOLDS")
display(triad_threshold_report_ch3)

print("\n2. SEGMENT SUMMARY")
display(triad_segment_report_ch3)

print("\n3. INDIVIDUAL TRIAD PROFILES — BUBBLE-CHART INPUT")
display(triad_chart_data_ch3)

print("\n4. TOP-THREE TRIAD FRAGRANCE EXAMPLES")
display(triad_examples_report_ch3)

print("\n5. VALIDATION RESULTS")
display(pd.DataFrame(triad_checks_ch3))


In [0]:

# =============================================================================
# CHAPTER 3 — FRAGRANCE REPRESENTATION AND SEGMENT CONSISTENCY
# =============================================================================

import pandas as pd

# ---------------------------------------------------------------------------
# 1. Prepare fragrance-level records for each analytical level
# ---------------------------------------------------------------------------

def prepare_level_comparison(assignments, segmented, profile_col, prefix):

    assignment_data = assignments.select(
        "id",
        "gender_clean",
        profile_col
    ).toPandas()

    profile_data = segmented[
        ["gender_clean", profile_col, "segment"]
    ].copy()

    result = assignment_data.merge(
        profile_data,
        on=["gender_clean", profile_col],
        how="inner",
        validate="many_to_one"
    )

    result = result.rename(columns={
        profile_col: f"{prefix}_profile",
        "segment": f"{prefix}_segment"
    })

    return result[
        [
            "id",
            "gender_clean",
            f"{prefix}_profile",
            f"{prefix}_segment"
        ]
    ]


dominant_fragrances_ch3 = prepare_level_comparison(
    dominant_assignments_ch3,
    dominant_segmented_ch3,
    "dominant_accord",
    "dominant"
)

dyad_fragrances_ch3 = prepare_level_comparison(
    df_dyad_assignments_ch3,
    dyad_segmented_ch3,
    "dyad",
    "dyad"
)

triad_fragrances_ch3 = prepare_level_comparison(
    df_triad_assignments_ch3,
    triad_segmented_ch3,
    "triad",
    "triad"
)

# ---------------------------------------------------------------------------
# 2. Match the same fragrances across analytical levels
# ---------------------------------------------------------------------------

fragrance_level_comparison_ch3 = (
    dominant_fragrances_ch3
    .merge(
        dyad_fragrances_ch3,
        on=["id", "gender_clean"],
        how="outer",
        validate="one_to_one"
    )
    .merge(
        triad_fragrances_ch3,
        on=["id", "gender_clean"],
        how="outer",
        validate="one_to_one"
    )
)

SEGMENT_COLUMNS = [
    "dominant_segment",
    "dyad_segment",
    "triad_segment"
]

fragrance_level_comparison_ch3["levels_present"] = (
    fragrance_level_comparison_ch3[SEGMENT_COLUMNS]
    .notna()
    .sum(axis=1)
)

# ---------------------------------------------------------------------------
# 3. Compare segment consistency
# ---------------------------------------------------------------------------

def compare_segments(row):
    segments = row[SEGMENT_COLUMNS].dropna().unique()

    if len(segments) < 2:
        return "Only one level"

    if len(segments) == 1:
        return "Consistent"

    return "Different"


fragrance_level_comparison_ch3["segment_consistency"] = (
    fragrance_level_comparison_ch3.apply(
        compare_segments,
        axis=1
    )
)

# ---------------------------------------------------------------------------
# 4. Fragrance representation across analytical levels
# ---------------------------------------------------------------------------

fragrance_level_coverage_ch3 = (
    fragrance_level_comparison_ch3
    .groupby(
        ["gender_clean", "levels_present"],
        as_index=False
    )
    .agg(
        fragrance_count=("id", "nunique")
    )
)

fragrance_level_coverage_ch3["coverage_pct"] = (
    fragrance_level_coverage_ch3["fragrance_count"]
    / fragrance_level_coverage_ch3.groupby("gender_clean")[
        "fragrance_count"
    ].transform("sum")
    * 100
).round(2)

# ---------------------------------------------------------------------------
# 5. Segment consistency across available levels
# ---------------------------------------------------------------------------

fragrance_segment_consistency_ch3 = (
    fragrance_level_comparison_ch3
    .groupby(
        [
            "gender_clean",
            "levels_present",
            "segment_consistency"
        ],
        as_index=False
    )
    .agg(
        fragrance_count=("id", "nunique")
    )
)

fragrance_segment_consistency_ch3["share_pct"] = (
    fragrance_segment_consistency_ch3["fragrance_count"]
    / fragrance_segment_consistency_ch3.groupby(
        ["gender_clean", "levels_present"]
    )["fragrance_count"].transform("sum")
    * 100
).round(2)

# ---------------------------------------------------------------------------
# 6. Pairwise segment comparisons
# ---------------------------------------------------------------------------

pair_definitions = [
    ("Dominant accords vs Dyads", "dominant_segment", "dyad_segment"),
    ("Dominant accords vs Triads", "dominant_segment", "triad_segment"),
    ("Dyads vs Triads", "dyad_segment", "triad_segment")
]

pairwise_results = []

for comparison_name, column_a, column_b in pair_definitions:

    pair_data = fragrance_level_comparison_ch3[
        [
            "id",
            "gender_clean",
            column_a,
            column_b
        ]
    ].dropna(subset=[column_a, column_b]).copy()

    pair_data["comparison"] = comparison_name

    pair_data["segment_comparison"] = (
        pair_data[column_a] == pair_data[column_b]
    ).map({
        True: "Same segment",
        False: "Different segments"
    })

    pairwise_results.append(pair_data)

pairwise_segment_comparison_ch3 = pd.concat(
    pairwise_results,
    ignore_index=True
)

pairwise_segment_summary_ch3 = (
    pairwise_segment_comparison_ch3
    .groupby(
        ["gender_clean", "comparison", "segment_comparison"],
        as_index=False
    )
    .agg(
        fragrance_count=("id", "nunique")
    )
)

pairwise_segment_summary_ch3["share_pct"] = (
    pairwise_segment_summary_ch3["fragrance_count"]
    / pairwise_segment_summary_ch3.groupby(
        ["gender_clean", "comparison"]
    )["fragrance_count"].transform("sum")
    * 100
).round(2)

# ---------------------------------------------------------------------------
# 7. Validation
# ---------------------------------------------------------------------------

assert fragrance_level_comparison_ch3[
    ["id", "gender_clean"]
].duplicated().sum() == 0

assert fragrance_level_comparison_ch3[
    "levels_present"
].between(1, 3).all()

print("Validation checks passed.")
print(
    f"Fragrances represented in at least one eligible level: "
    f"{fragrance_level_comparison_ch3['id'].nunique():,}"
)

# ---------------------------------------------------------------------------
# 8. Results
# ---------------------------------------------------------------------------

print("\n1. FRAGRANCE COVERAGE ACROSS ANALYTICAL LEVELS")
display(
    fragrance_level_coverage_ch3.sort_values(
        ["gender_clean", "levels_present"]
    )
)

print("\n2. SEGMENT CONSISTENCY ACROSS AVAILABLE LEVELS")
display(
    fragrance_segment_consistency_ch3.sort_values(
        ["gender_clean", "levels_present", "segment_consistency"]
    )
)

print("\n3. PAIRWISE SEGMENT COMPARISON")
display(
    pairwise_segment_summary_ch3.sort_values(
        ["gender_clean", "comparison", "segment_comparison"]
    )
)

print("\n4. FRAGRANCE-LEVEL DETAIL")
display(fragrance_level_comparison_ch3)


In [0]:
display(
    gender_level_comparison_ch3.sort_values(
        ["gender_clean", "analytical_level"]
    )
)

In [0]:
for name in sorted(globals()):
    obj = globals()[name]
    if (
        ("dyad" in name.lower() or "triad" in name.lower())
        and isinstance(obj, pd.DataFrame)
    ):
        print(f"\n{name}: {obj.shape}")
        print(obj.columns.tolist())